# Imports

In [1]:
import sys
sys.path.append('../src')

import pymongo
import pandas as pd
from load_to_mongodb import db

In [2]:
# Print to see if db was well imported from "load_to_mongodb.py"
print(db.movies_collection.count_documents({}))

1478


# Q1

In [3]:
pipeline = [
    {
        "$match": {
            "trailer.genre": {
                "$in": ["Action", "Comedy"]
            }
        }
    },
    {
        "$project": {
            "_id": 0,
            "genre": "$trailer.genre",
            "favorability": "$trailer.favorability",
            "averageRating": "$ratings.averageRating"
        }
    }
]

q1_result = db.movies_collection.aggregate(pipeline)

list(q1_result)[:5]

[{'genre': 'Action', 'favorability': 0.6518987341772152, 'averageRating': 5.7},
 {'genre': 'Action', 'favorability': 0.631578947368421, 'averageRating': 7.7},
 {'genre': 'Comedy', 'favorability': 0.6506024096385542, 'averageRating': 5.1},
 {'genre': 'Action', 'favorability': 0.5481481481481482, 'averageRating': 5.3},
 {'genre': 'Comedy', 'favorability': 0.6, 'averageRating': 5.8}]

# Q2

In [4]:
pipeline = [
    {
        "$match": {
            "$or": [
                {
                    "ratings.averageRating": {
                        "$gte": 7
                    }
                },
                {
                    "ratings.averageRating": {
                        "$lte": 4
                    }
                }
            ]
        }
    },
    {
        "$addFields": {
            "rating_group": {
                "$cond": [
                    {
                        "$gte": ["$ratings.averageRating", 7]
                    },
                    "high rating",
                    "low rating"
                ]
            }
        }
    },
    {
        "$project": {
            "_id": 0,
            "primaryTitle": 1,
            "averageRating": "$ratings.averageRating",
            "favorability": "$trailer.favorability",
            "rating_group": 1
        }
    },
    {
        "$sort": {
            "rating_group": 1,
            "averageRating": -1,
            "favorability": -1
        }
    }
]

q2_result = db.movies_collection.aggregate(pipeline)

list(q2_result)[:5]

[{'primaryTitle': 'Last Days',
  'rating_group': 'high rating',
  'averageRating': 9.2,
  'favorability': 0.5511363636363636},
 {'primaryTitle': 'Trial by Fire',
  'rating_group': 'high rating',
  'averageRating': 9.2,
  'favorability': 0.4675324675324675},
 {'primaryTitle': 'Waiting',
  'rating_group': 'high rating',
  'averageRating': 9.0,
  'favorability': 0.7666666666666667},
 {'primaryTitle': 'Not Fade Away',
  'rating_group': 'high rating',
  'averageRating': 9.0,
  'favorability': 0.7142857142857143},
 {'primaryTitle': 'The Island',
  'rating_group': 'high rating',
  'averageRating': 9.0,
  'favorability': 0.6730769230769231}]

In [5]:
pipeline = [
    {
        "$match": {
            "trailer.rating": {"$in": ["R", "PG", "PG-13", "G"]}
        }
    },
    {
        "$addFields": {
            "rating_group": {
                "$cond": {
                    "if": {"$eq": ["$trailer.rating", "R"]},
                    "then": "R-Rated",
                    "else": "Non R-Rated"
                }
            }
        }
    },
    {
        "$project": {
            "_id": 0,
            "primaryTitle": 1,
            "rating": "$trailer.rating",
            "rating_group": 1,
            "averageRating": "$ratings.averageRating",
            "favorability": "$trailer.favorability"
        }
    },
    {
        "$sort": {
            "rating_group": 1,
            "averageRating": 1
        }
    }
]

q3_result = db.movies_collection.aggregate(pipeline)
list(q3_result)[:5]


[{'primaryTitle': 'LOL',
  'rating_group': 'Non R-Rated',
  'rating': 'PG-13',
  'averageRating': 1.5,
  'favorability': 0.7987012987012987},
 {'primaryTitle': 'Disaster Movie',
  'rating_group': 'Non R-Rated',
  'rating': 'PG-13',
  'averageRating': 1.9,
  'favorability': 0.4034090909090909},
 {'primaryTitle': 'Into the Woods',
  'rating_group': 'Non R-Rated',
  'rating': 'PG',
  'averageRating': 2.1,
  'favorability': 0.6705202312138728},
 {'primaryTitle': 'Son of the Mask',
  'rating_group': 'Non R-Rated',
  'rating': 'PG',
  'averageRating': 2.3,
  'favorability': 0.5833333333333334},
 {'primaryTitle': 'Dragonball Evolution',
  'rating_group': 'Non R-Rated',
  'rating': 'PG',
  'averageRating': 2.5,
  'favorability': 0.6264367816091954}]